# Занятие 35. Решающее дерево

**Решающее дерево** (decision tree) — модель, которая принимает решение цепочкой вопросов «да / нет». Каждый вопрос смотрит на один признак и сравнивает его с порогом.

На занятии 33 вы уже видели, что сложная модель без ограничений переобучается. Дерево — яркий пример: оно легко строит нелинейные правила и понятно человеку, но глубокое дерево запоминает шум.

**Сквозной пример занятия:** мини-датасет «подготовка к экзамену» (часы подготовки, часы сна → сдал / не сдал). Все определения и демо-код ниже — про него.

## 1. Правила «если — то» и устройство дерева

**Определение.** Решающее дерево — это набор вложенных правил вида:

> если признак ≤ порог, то идём влево, иначе вправо; в листе — итоговый прогноз.

**Части дерева:**
- **Корень** (root) — первый вопрос, с которого начинается прогноз.
- **Внутренний узел** — промежуточный вопрос; у него есть дочерние узлы.
- **Ветвь** — ответ «да» или «нет» на вопрос узла.
- **Лист** (leaf) — узел без дочерних вопросов; здесь хранится прогноз.

**Пример на сквозном датасете:** «подготовка ≤ 4 часов?» → «сон ≤ 6 часов?» → прогноз «не сдал».

**Как читать `plot_tree`:** в каждом прямоугольнике обычно написано:
- условие (`подготовка <= 3.5`);
- `gini` — неоднородность классов в узле (0 = все одного класса);
- `samples` — сколько учебных объектов попало в узел;
- `value` — сколько объектов каждого класса;
- `class` — прогноз класса в листе.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

X=pd.DataFrame({'подготовка':[1,2,3,4,5,6,7,8,4,6], 'сон':[5,8,6,7,5,8,6,9,9,4]})
y=np.array([0,0,0,0,1,1,1,1,1,0])
tree=DecisionTreeClassifier(max_depth=3,random_state=42).fit(X,y)
plt.figure(figsize=(12,6)); plot_tree(tree,feature_names=X.columns,class_names=['не сдал','сдал'],filled=True); plt.show()


## 2. Как выбирается признак и порог

**Признак** — столбец, по которому задаётся вопрос (`подготовка` или `сон`).

**Порог** — число $t$ в условии «признак ≤ $t$». Для признака «подготовка» порог 4.5 означает вопрос «подготовка ≤ 4.5?».

**Неоднородность (impurity)** — насколько в узле смешаны классы. Чистый узел (все «сдал» или все «не сдал») — хороший кандидат в лист. Смешанный узел — кандидат на новый вопрос.

**Как ищет sklearn:** для каждого признака перебираются пороги между соседними **отсортированными** уникальными значениями (часто — середины между ними). Для каждого кандидата считается, насколько улучшится однородность дочерних узлов.

**Критерий** (`criterion` в `DecisionTreeClassifier`) — формула неоднородности: по умолчанию `gini`, альтернатива — `entropy` (п. 5).

## 3. Gini impurity

**Gini** измеряет, насколько узел «смешанный»:

$$Gini = 1 - \sum_k p_k^2,$$

где $p_k$ — доля класса $k$ среди объектов узла.

| Состав узла | $p_0$, $p_1$ | Gini |
|-------------|--------------|------|
| 4 объекта класса 1 | 0, 1 | 0.0 (чистый) |
| 2+2 | 0.5, 0.5 | 0.5 (максимально смешанный для 2 классов) |
| 3×0 и 1×1 | 0.75, 0.25 | 0.375 |

Gini = 0, если в узле один класс. Чем сильнее смешение, тем Gini больше.

In [ ]:
def gini(labels):
 counts=np.bincount(labels); p=counts/counts.sum(); return 1-np.sum(p**2)

for labels in [np.array([1,1,1,1]),np.array([0,0,1,1]),np.array([0,0,0,1])]:
 print(labels,'Gini =',round(gini(labels),3))


## 4. Information gain и качество разбиения

После вопроса объекты делятся на **левый** и **правый** дочерний узел. Считают взвешенный Gini после разбиения:

$$Gini_{split}=\frac{n_L}{n}Gini_L+\frac{n_R}{n}Gini_R.$$

**Information gain** (прирост информации) — насколько уменьшилась неоднородность:

$$gain = Gini_{parent} - Gini_{split}.$$

Алгоритм **жадно** выбирает пару (признак, порог) с **максимальным** gain.

**Ручной пример.** Родительский узел: 6 объектов, классы `[0,0,0,1,1,1]`, $Gini=0.5$.
- Разбиение A: слева `[0,0,0]`, справа `[1,1,1]` → оба листа чистые → $Gini_{split}=0$ → $gain=0.5$.
- Разбиение B: слева `[0,0]`, справа `[0,1,1,1]` → смешение справа → gain меньше.

Выбираем разбиение A.

In [ ]:
parent=np.array([0,0,0,1,1,1])
splits={
 'A: классы разделены':(np.array([0,0,0]),np.array([1,1,1])),
 'B: классы смешаны':(np.array([0,0]),np.array([0,1,1,1])),
}

def weighted_gini(left,right):
 n=len(left)+len(right)
 return (len(left)*gini(left)+len(right)*gini(right))/n

parent_g=gini(parent)
for name,(L,R) in splits.items():
 split_g=weighted_gini(L,R)
 print(name,'Gini_split =',round(split_g,3),'gain =',round(parent_g-split_g,3))


## 5. Энтропия *

**Энтропия** — альтернативный критерий неоднородности:

$$Entropy = -\sum_k p_k \log_2 p_k.$$

Смысл тот же: в чистом узле энтропия 0, при смешении растёт. Хороший вопрос уменьшает энтропию (аналог gain для Gini).

| Узел | Gini | Entropy |
|------|------|---------|
| [0,0,0,0] | 0.000 | 0.000 |
| [0,0,1,1] | 0.500 | 1.000 |
| [0,0,0,1] | 0.375 | 0.811 |

В sklearn: `DecisionTreeClassifier(criterion='entropy')`. На практике Gini и entropy часто дают похожие деревья; важнее ограничить глубину и размер листьев (п. 10).

## 6. Рекурсивное построение и остановка

**Рекурсия:** после первого вопроса алгоритм обрабатывает левую и правую группы **отдельно** — снова ищет лучший (признак, порог). Так повторяется, пока не сработает **правило остановки**.

**Когда построение прекращается:**
- узел **чистый** (один класс);
- объектов в узле **мало** (`min_samples_split`, `min_samples_leaf`);
- достигнута **максимальная глубина** (`max_depth`).

Параметры остановки подробнее — в п. 10. Сейчас важно: дерево строится **сверху вниз**, на каждом шаге локально лучший вопрос.

## 7. Прогноз в листе: класс, вероятность, регрессия

**Классификация.** В листе дерево хранит доли классов среди учебных объектов этого листа:
- `predict` — самый частый класс;
- `predict_proba` — доли классов (оценка вероятности, но не всегда хорошо откалибрована).

**Размер листа.** Лист из 2 объектов одного класса даст вероятность 0 или 1 — это ненадёжно. Параметр `min_samples_leaf` заставляет накапливать больше объектов перед прогнозом долей.

**Регрессия** (`DecisionTreeRegressor`). В листе прогноз — **среднее** целевой переменной среди объектов листа. Разбиения уменьшают **MSE** (средний квадрат ошибки), а не Gini.

In [ ]:
prices_in_leaf=np.array([8.,10.,12.])
print('Прогноз регрессионного листа (среднее):',prices_in_leaf.mean())


In [ ]:
new=pd.DataFrame({'подготовка':[3,7],'сон':[8,5]})
print('Классы:',tree.predict(new))
print('Доли классов в листьях:')
print(tree.predict_proba(new).round(3))


## 8. Путь одного объекта

**Путь** — последовательность узлов от корня до листа для конкретного объекта.

Для объекта с `подготовка=3`, `сон=8` дерево сначала проверяет корневой вопрос, затем — следующий узел по ветви «да» или «нет», пока не дойдёт до листа.

`decision_path` показывает, через какие узлы прошёл объект. `export_text` печатает те же правила в виде текста «если — то».

In [ ]:
# Пройдём первый новый объект по его пути.
path=tree.decision_path(new.iloc[[0]])
for node in path.indices:
 feature=tree.tree_.feature[node]
 if feature < 0:
  print(f'Узел {node}: лист, прогноз {tree.predict(new.iloc[[0]])[0]}')
 else:
  value=new.iloc[0,feature]; threshold=tree.tree_.threshold[node]
  sign='<=' if value<=threshold else '>'
  print(f'Узел {node}: {X.columns[feature]}={value} {sign} {threshold:.2f}')
print()
print(export_text(tree,feature_names=list(X.columns)))


## 9. Ступенчатая и осевая граница решений

Каждый вопрос вида $x_j \le t$ делит плоскость признаков **вертикальной или горизонтальной** линией. Несколько вопросов создают прямоугольные области — **ступенчатую** границу.

Дерево строит **нелинейную**, но **осевую** границу: линии параллельны осям признаков. Наклонную границу дерево приближает «лестницей» из прямоугольников — чем точнее, тем больше узлов и выше риск переобучения.

In [ ]:
x1=np.linspace(0,9,200); x2=np.linspace(3,10,200); xx1,xx2=np.meshgrid(x1,x2)
grid=pd.DataFrame({'подготовка':xx1.ravel(),'сон':xx2.ravel()})
surface=tree.predict_proba(grid)[:,1].reshape(xx1.shape)
plt.contourf(xx1,xx2,surface,levels=[0,.5,1],cmap='RdYlGn',alpha=.35)
plt.scatter(X['подготовка'],X['сон'],c=y,cmap='RdYlGn',edgecolor='black')
plt.xlabel('Подготовка'); plt.ylabel('Сон'); plt.title('Ступенчатая граница дерева'); plt.show()


## 10. Переобучение, гиперпараметры и обрезка

**Симптом переобучения:** глубокое дерево выделяет отдельный лист почти под каждый объект train. Train-accuracy → 1.0, validation почти не растёт — модель запомнила шум.

**Предварительное ограничение** (не даём дереву разрастись):
- `max_depth` — максимальная глубина;
- `min_samples_split` — минимум объектов в узле, чтобы сделать новое разбиение;
- `min_samples_leaf` — минимум объектов в листе.

**Post-pruning (обрезка после построения):** сначала строится полное дерево, затем удаляются ветви с малой пользой. В sklearn силу обрезки задаёт `ccp_alpha`: чем больше `ccp_alpha`, тем короче дерево.

Все параметры выбирают по **validation** или кросс-валидации (занятие 33).

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

Xm,ym=make_moons(n_samples=350,noise=.3,random_state=42)
Xt,Xv,yt,yv=train_test_split(Xm,ym,test_size=.35,stratify=ym,random_state=42)
for d in [1,3,6,None]:
 m=DecisionTreeClassifier(max_depth=d,random_state=42).fit(Xt,yt)
 print('depth',d,'train',round(accuracy_score(yt,m.predict(Xt)),3),'validation',round(accuracy_score(yv,m.predict(Xv)),3))


In [ ]:
for alpha in [0.0,0.005,0.02,0.08]:
 pruned=DecisionTreeClassifier(ccp_alpha=alpha,random_state=42).fit(Xt,yt)
 print('ccp_alpha',alpha,'листьев',pruned.get_n_leaves(),'глубина',pruned.get_depth())


## 11. Масштаб, пропуски и категории в sklearn

**Масштабирование.** Дерево сравнивает значение с порогом, а не расстояние между признаками. `StandardScaler` обычно **не нужен**: порядок значений сохраняется. Строго монотонное преобразование (например, log) меняет пороги, но не порядок.

**Пропуски (`NaN`).** В новых версиях sklearn дерево может направлять объекты с пропуском в левую или правую ветвь. Перед проектом проверьте документацию своей версии и зафиксируйте стратегию (imputation vs native NaN).

**Категории.** `DecisionTreeClassifier` не принимает строковые категории напрямую. Используйте `OneHotEncoder` (без ложного порядка) или осмысленное кодирование. NaN ≠ категория: категориальные признаки кодируют отдельно.

## 12. Важность признаков: impurity и permutation

**Impurity importance** (`feature_importances_`) — сумма gain по всем разбиениям, где использовался признак. Удобно для первого взгляда, но:
- не доказывает причинность;
- завышает признаки с множеством уникальных значений;
- делится между коррелирующими признаками.

**Permutation importance** — на validation перемешивают один столбец и смотрят, насколько упало качество. Показывает, использовала ли модель признак на новых данных. Коррелирующие признаки могут «подменять» друг друга — падение качества будет маленьким у каждого по отдельности.

In [ ]:
pd.Series(tree.feature_importances_,index=X.columns).sort_values(ascending=False)


In [ ]:
from sklearn.inspection import permutation_importance

perm=permutation_importance(tree,X,y,n_repeats=10,random_state=42)
pd.DataFrame({'impurity':tree.feature_importances_,'permutation':perm.importances_mean},index=X.columns)


## 13. Плюсы, ограничения и нестабильность

**Плюсы:**
- нелинейные правила без ручного feature engineering;
- **взаимодействия** признаков (например, «мало сна **и** мало подготовки») возникают автоматически;
- понятные правила «если — то»;
- масштабирование обычно не нужно.

**Ограничения:**
- **нестабильность** — небольшое изменение train может поменять первый вопрос и перестроить всё дерево;
- ступенчатые прогнозы, сильное переобучение без ограничений;
- **слабая экстраполяция** — за пределами диапазона train дерево продолжает давать константу из ближайшего листа.

> **Главная мысль.** Дерево повторяет один шаг: выбирает вопрос, делит данные, рекурсивно строит поддеревья. Сложность нужно ограничивать. На следующем занятии (bagging / random forest) много нестабильных деревьев усредняют — правила становятся устойчивее.